# 3. Parsing statistics

*Author: La Anh Hào · code: `src/13_dedup_export.ipynb`, `src/15_run_parsing.ipynb`*

This chapter reports what the parser produced from the corpus of Milestone 1: how many versions it could
parse, how much of each paper's history is covered, and how many elements and BibTeX entries remain after the
two deduplication steps. Every number is read from the files that `src/15_run_parsing.ipynb` writes at the
end of the run (`m2_stats.json`, `m2_parse_status.csv` with one row per version, and the per-paper summaries
of its resume index), copied to `data/` of the report repository.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import report_data as rd

rd.check()
rd.style()
stats = rd.stats()
versions, papers = rd.versions(), rd.papers()
status = pd.Series({k: int(stats["status"].get(k, 0)) for k in rd.STATUSES})
with_tex = int(stats["versions_with_tex"])
rd.md(f"Parser version `{stats['parser_version']}`; statistics computed on "
      f"{stats['computed_at'][:16].replace('T', ' ')} UTC over **{stats['papers']:,}** papers and "
      f"**{stats['versions']:,}** versions.")

## 3.1 Parse success rate

**Definition.** The statement asks for every version of every paper, including the versions without LaTeX,
which stay in the corpus as empty records. Each version listed in the paper's `metadata.json` ends with one
status:

- `ok`: a root file was found, the document tree was built and its rows were written;
- `empty_source`: the version has no `.tex` file (a PDF-only submission, or a source that Milestone 1 could
  not download); it is recorded, with no rows;
- `no_root`: the version has `.tex` files but none of them is a LaTeX document (`\documentclass`);
- `error`: an exception in one of the steps; the message is logged and the other versions go on.

The **parse success rate** is the share of `ok` among the versions that have LaTeX source (`ok`, `no_root`
and `error`). The share among all versions is given too; the difference is the PDF-only versions, which no
parser can read.

In [ ]:
labels = {"ok": "`ok`", "empty_source": "`empty_source`", "no_root": "`no_root`", "error": "`error`"}
t31 = pd.DataFrame({
    "Status": [labels[k] for k in rd.STATUSES],
    "Versions": status.values,
    "Share of all versions": 100 * status.values / stats["versions"],
    "Share of versions with LaTeX": [np.nan if k == "empty_source" else 100 * v / with_tex for k, v in status.items()],
})
rd.table(t31, "Table 3.1 – Status of every version listed in metadata.json",
         {"Share of all versions": "{:.2f}%", "Share of versions with LaTeX": "{:.2f}%"})
rd.md(f"**Parse success rate: {stats['parse_success_rate_of_tex_versions_pct']:.2f}%** of the {with_tex:,} versions "
      f"with LaTeX source ({status['ok']:,} versions), and {stats['parse_success_rate_of_all_versions_pct']:.2f}% of "
      f"all {stats['versions']:,} versions. "
      + ("No version failed with an exception." if status["error"] == 0
         else f"{status['error']:,} versions failed with an exception (listed in `m2_parse_status.csv`)."))

In [ ]:
tex = versions["status"] != "empty_source"
t32 = (versions.assign(tex=tex, ok=versions["status"] == "ok", no_root=versions["status"] == "no_root",
                       error=versions["status"] == "error")
       .groupby("range")
       .agg(papers=("arxiv_id", "nunique"), versions=("status", "size"), tex=("tex", "sum"), ok=("ok", "sum"),
            no_root=("no_root", "sum"), error=("error", "sum")))
t32["rate"] = 100 * t32["ok"] / t32["tex"]
t32.loc["all"] = t32.sum(numeric_only=True)
t32.loc["all", "rate"] = 100 * t32.loc["all", "ok"] / t32.loc["all", "tex"]
t32 = t32.reset_index().rename(columns={
    "range": "Range", "papers": "Papers", "versions": "Versions", "tex": "With LaTeX", "ok": "ok",
    "no_root": "no_root", "error": "error", "rate": "Success rate"})
rd.table(t32, "Table 3.2 – Parse success rate per ID range (R1–R4 as in chapter 1 of Milestone 1)",
         {"Success rate": "{:.2f}%"})

In [ ]:
order = ["ok", "no_root", "error", "empty_source"]
colors = {"ok": rd.BLUE, "no_root": rd.ORANGE, "error": rd.MAGENTA, "empty_source": rd.AXIS}
counts = versions.groupby(["range", "status"]).size().unstack(fill_value=0).reindex(columns=order, fill_value=0)
share = 100 * counts.div(counts.sum(axis=1), axis=0)
fig, ax = plt.subplots(figsize=(9, 2.6))
left = np.zeros(len(share))
for name in order:
    ax.barh(share.index, share[name], left=left, color=colors[name], height=0.55, label=name,
            edgecolor=rd.SURFACE, linewidth=1)
    left += share[name].values
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("Share of the versions of the range (%)")
ax.grid(axis="y", visible=False)
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.32))
ax.set_title("Status of the versions per ID range")
plt.tight_layout()
plt.show()
rd.figure_caption("Figure 3.1 – Status of the versions of each range. The grey part is the versions without "
                  "LaTeX source; `no_root` and `error` are too few to be visible.")

### Versions without a root

The statement defines the root as the `.tex` file that holds `\documentclass`. A version whose files never
declare a document class has no root and is not parsed. In the run of parser 13.8 these versions were
checked one by one: they are **Plain TeX** papers (`\magnification`, `\input` of macro files, `\bye` or
`\end` at the end), which never were LaTeX documents. Before 13.8, two more versions were in this list
because their files end lines with a carriage return only (an old Mac format): read as one single line, a `%`
on the first line turned the whole file into a comment (section [2.1](hierarchy.md)).

In [ ]:
no_root = versions.loc[versions["status"] == "no_root", ["arxiv_id", "version", "error"]]
if len(no_root):
    rd.table(no_root.rename(columns={"arxiv_id": "Paper", "version": "Version", "error": "Logged reason"}),
             f"Table 3.3 – The {len(no_root)} versions with LaTeX files but no root")
else:
    rd.md("Every version with LaTeX files has a root.")

### Versions without LaTeX source

These versions are part of the corpus but have nothing to parse. The reason is taken from the download
receipts of Milestone 1.

In [ ]:
reason_labels = {
    "pdf_only": "arXiv has no LaTeX source for the version (PDF only)",
    "files_missing": "a download receipt exists but the vN/ folder is not on Drive",
    "not_found": "the source request returned HTTP 404",
    "not_downloaded": "no download receipt",
    "": "no reason logged",
}
reasons = pd.Series(stats.get("empty_source_reasons") or {}, dtype="int64").sort_values(ascending=False)
t34 = pd.DataFrame({"Reason": [reason_labels.get(k, k) for k in reasons.index], "Versions": reasons.values,
                    "Share of all versions": 100 * reasons.values / stats["versions"]})
rd.table(t34, f"Table 3.4 – Why {status['empty_source']:,} versions have no LaTeX source",
         {"Share of all versions": "{:.2f}%"})

## 3.2 Version coverage of every paper

**Definition.** The *version coverage* of a paper is the number of its versions that were parsed over the
number of versions it has on arXiv (`metadata.json`). A paper is *fully covered* when every version was
parsed, *partly covered* when some were, and *not covered* when none was, which is the case of a paper whose
versions are all PDF-only.

In [ ]:
dist = stats["version_coverage"]["distribution"]
cov = pd.DataFrame([(int(key.split("/")[1]), int(key.split("/")[0]), n) for key, n in dist.items()],
                   columns=["versions", "parsed", "papers"])
cov["group"] = np.select([cov["parsed"] == cov["versions"], cov["parsed"] == 0], ["full", "none"], "partial")
cov["n"] = cov["versions"].clip(upper=5)
by_n = cov.pivot_table(index="n", columns="group", values="papers", aggfunc="sum", fill_value=0)
by_n = by_n.reindex(columns=["full", "partial", "none"], fill_value=0)
names = {n: (f"{n}" if n < 5 else "5 or more") for n in by_n.index}
share = 100 * by_n.div(by_n.sum(axis=1), axis=0)
fig, ax = plt.subplots(figsize=(9, 3.2))
bottom = np.zeros(len(share))
xs = [names[n] for n in share.index]
for group, color in (("full", rd.BLUE), ("partial", rd.YELLOW), ("none", rd.AXIS)):
    ax.bar(xs, share[group], bottom=bottom, color=color, width=0.6, label=group, edgecolor=rd.SURFACE, linewidth=1)
    bottom += share[group].values
for x, total in zip(xs, by_n.sum(axis=1)):
    ax.text(x, 101.5, f"{total:,} papers", ha="center", va="bottom", fontsize=8, color="#52514e")
ax.set_ylim(0, 112)
ax.set_xlabel("Versions of the paper on arXiv")
ax.set_ylabel("Share of the papers (%)")
ax.grid(axis="x", visible=False)
ax.legend(title="versions parsed", loc="lower left", bbox_to_anchor=(1.0, 0.0))
ax.set_title("Version coverage per paper")
plt.tight_layout()
plt.show()
rd.figure_caption("Figure 3.2 – For the papers with 1, 2, ... versions: share whose versions were all parsed "
                  "(full), some (partial) or none. The number of papers is written above each bar.")

In [ ]:
t35 = by_n.copy()
t35["papers"] = t35.sum(axis=1)
t35["full_pct"] = 100 * t35["full"] / t35["papers"]
t35 = t35.reset_index()
t35["n"] = t35["n"].map(names)
t35 = t35.rename(columns={"n": "Versions on arXiv", "papers": "Papers", "full": "Every version parsed",
                          "partial": "Some versions", "none": "No version", "full_pct": "Fully covered"})
t35 = t35[["Versions on arXiv", "Papers", "Every version parsed", "Some versions", "No version", "Fully covered"]]
rd.table(t35, "Table 3.5 – Version coverage by number of versions", {"Fully covered": "{:.2f}%"})
c = stats["version_coverage"]
rd.md(f"**{c['papers_full']:,}** papers ({rd.pct(c['papers_full'], stats['papers'])}) have every version parsed, "
      f"**{c['papers_partial']:,}** some of them and **{c['papers_none']:,}** none. Over all papers, a paper has "
      f"**{c['mean_pct']:.2f}%** of its versions parsed on average.")

## 3.3 Size of the parsed documents

The size of a version is the number of nodes of its tree, from the `document` root to the sentences,
formulas and figures (chapter [2.1](hierarchy.md)).

In [ ]:
ok = versions[versions["status"] == "ok"]
nodes = ok["n_nodes"].clip(lower=1)
fig, ax = plt.subplots(figsize=(9, 3))
bins = np.logspace(0, np.log10(nodes.max()) + 0.05, 60)
ax.hist(nodes, bins=bins, color=rd.BLUE, edgecolor=rd.SURFACE, linewidth=0.5)
ax.set_xscale("log")
ax.axvline(nodes.median(), color=rd.ORANGE, lw=1.5)
ax.text(nodes.median() * 1.08, ax.get_ylim()[1] * 0.92, f"median {nodes.median():,.0f}", color=rd.ORANGE, fontsize=9)
ax.set_xlabel("Nodes in the tree of one version (log scale)")
ax.set_ylabel("Versions")
ax.grid(axis="x", visible=False)
ax.set_title("Size of the tree of each parsed version")
plt.tight_layout()
plt.show()
rd.figure_caption("Figure 3.3 – Number of tree nodes per parsed version.")
q = nodes.quantile([0.1, 0.5, 0.9, 0.99])
rd.md(f"A parsed version has **{nodes.median():,.0f}** nodes at the median (10% of the versions have fewer "
      f"than {q[0.1]:,.0f}, 10% more than {q[0.9]:,.0f}; the largest has {nodes.max():,}). "
      f"{int((ok['n_nodes'] <= 1).sum()):,} parsed versions have a tree with the document node only.")

## 3.4 Elements before and after deduplication

**Definitions.** *Before deduplication* counts every node of every version's tree. *After deduplication*
counts the rows of the `elements` table, one per distinct element of a paper over all of its versions
(chapter [2.4](deduplication.ipynb)): a sentence, formula or figure whose cleaned content is identical in two
versions, or twice in one version, is one element with several `hierarchy` rows. A section is shared when
it has the same title under the same parent.

In [ ]:
before = pd.Series(stats["elements_before_dedup_by_kind"], dtype="int64")
after = pd.Series(stats["elements_after_dedup_by_kind"], dtype="int64")
kinds = [k for k in rd.KINDS if k in before.index or k in after.index]
kinds += [k for k in before.index.union(after.index) if k not in kinds]
t36 = pd.DataFrame({"Kind": [f"`{k}`" for k in kinds],
                    "Before (nodes)": before.reindex(kinds).fillna(0).astype(int).values,
                    "After (elements)": after.reindex(kinds).fillna(0).astype(int).values})
t36["Removed"] = 100 * (1 - t36["After (elements)"] / t36["Before (nodes)"].where(t36["Before (nodes)"] > 0))
total = pd.DataFrame({"Kind": ["**all kinds**"], "Before (nodes)": [stats["elements_before_dedup"]],
                      "After (elements)": [stats["elements_after_dedup"]],
                      "Removed": [100 * (1 - stats["elements_after_dedup"] / stats["elements_before_dedup"])]})
rd.table(pd.concat([t36, total], ignore_index=True), "Table 3.6 – Elements per kind before and after deduplication",
         {"Removed": "{:.1f}%"})

In [ ]:
y, height = np.arange(len(kinds)), 0.38
fig, ax = plt.subplots(figsize=(9, 0.36 * len(kinds) + 1.1))
ax.barh(y - height / 2, before.reindex(kinds).fillna(0).clip(lower=1), height, color=rd.BLUE,
        label="before: nodes of every version")
ax.barh(y + height / 2, after.reindex(kinds).fillna(0).clip(lower=1), height, color=rd.ORANGE,
        label="after: rows of elements")
ax.set_yticks(y, kinds)
ax.invert_yaxis()
ax.set_xscale("log")
ax.set_xlabel("Count (log scale)")
ax.grid(axis="y", visible=False)
ax.legend(loc="lower right")
ax.set_title("Elements per kind, before and after deduplication")
plt.tight_layout()
plt.show()
rd.figure_caption("Figure 3.4 – The same numbers as Table 3.6, on a log scale.")

The reduction depends on how many versions a paper has. With one version, only the repetitions inside the
paper are merged (a formula written twice, a sentence repeated in a caption and in the text, the same
untitled list); with two or more, every sentence, formula and figure that did not change between versions is
stored once.

In [ ]:
parsed = papers[papers["versions_ok"] > 0].copy()
parsed["n"] = parsed["versions_ok"].clip(upper=4)
t37 = parsed.groupby("n").agg(papers=("arxiv_id", "size"), nodes=("nodes", "sum"), elements=("elements", "sum"),
                              bib_before=("bib_before", "sum"), bib_after=("bib_after", "sum"))
t37["el_removed"] = 100 * (1 - t37["elements"] / t37["nodes"])
t37["bib_removed"] = 100 * (1 - t37["bib_after"] / t37["bib_before"].where(t37["bib_before"] > 0))
fig, ax = plt.subplots(figsize=(9, 2.8))
labels_n = [f"{n}" if n < 4 else "4 or more" for n in t37.index]
x = np.arange(len(t37))
b1 = ax.bar(x - 0.2, t37["el_removed"], 0.38, color=rd.BLUE, label="elements")
b2 = ax.bar(x + 0.2, t37["bib_removed"], 0.38, color=rd.ORANGE, label="BibTeX entries")
ax.bar_label(b1, fmt="%.1f%%", padding=2, fontsize=8)
ax.bar_label(b2, fmt="%.1f%%", padding=2, fontsize=8)
ax.set_xticks(x, labels_n)
ax.set_xlabel("Versions of the paper that were parsed")
ax.set_ylabel("Removed by deduplication (%)")
ax.set_ylim(0, max(t37[["el_removed", "bib_removed"]].max().max(), 1) * 1.25)
ax.grid(axis="x", visible=False)
ax.legend()
ax.set_title("Share removed by deduplication, by number of parsed versions")
plt.tight_layout()
plt.show()
rd.figure_caption("Figure 3.5 – Share of the nodes (blue) and of the extracted BibTeX entries (orange) that "
                  "deduplication removes, by number of parsed versions of the paper.")
t37 = t37.reset_index()
t37["n"] = labels_n
rd.table(t37.rename(columns={"n": "Parsed versions", "papers": "Papers", "nodes": "Nodes", "elements": "Elements",
                             "el_removed": "Elements removed", "bib_before": "BibTeX before",
                             "bib_after": "BibTeX after", "bib_removed": "BibTeX removed"}),
         "Table 3.7 – Deduplication by number of parsed versions",
         {"Elements removed": "{:.1f}%", "BibTeX removed": "{:.1f}%"})

## 3.5 BibTeX entries before and after deduplication

The references of a version come from one of three places (chapter [2.3](references.md)). The source is
counted over the versions with LaTeX source:

- `bib`: the `.bib` files named by `\bibliography{...}` or `\addbibresource{...}`;
- `bibitem`: a `thebibliography` list written in the `.tex` files, converted to BibTeX;
- `missing_bbl`: the paper declares a `.bib` file that its authors did not upload. arXiv compiled it into a
  `.bbl` file, which the download of Milestone 1 did not keep (only `.tex` and `.bib` were kept), so these
  references are lost;
- `none`: no bibliography at all.

In [ ]:
source_labels = {"bib": "`bib`: a .bib file", "bibitem": "`bibitem`: thebibliography in the .tex",
                 "bbl": "`bbl`: a .bbl file", "missing_bbl": "`missing_bbl`: .bib declared but not shipped",
                 "none": "`none`: no bibliography"}
src = pd.Series(stats["bib_source_of_tex_versions"], dtype="int64").sort_values(ascending=False)
t38 = pd.DataFrame({"Source": [source_labels.get(k, f"`{k}`") for k in src.index], "Versions": src.values,
                    "Share": 100 * src.values / src.sum()})
rd.table(t38, "Table 3.8 – Source of the references of the versions with LaTeX", {"Share": "{:.2f}%"})
b0, b1 = stats["bib_entries_before_dedup"], stats["bib_entries_after_dedup"]
rd.md(f"The versions yield **{b0:,}** BibTeX entries, summed over versions; after deduplication inside each "
      f"paper the `bibtex` table has **{b1:,}** rows ({rd.pct(b0 - b1, b0, 1)} fewer). An entry is merged with "
      f"another when they share the key in one version, the DOI, the arXiv ID or the normalised title (with "
      f"the rules of chapter [2.3](references.md)), and the merged row keeps the union of their fields. "
      f"Table 3.7 shows that most of the reduction comes from papers with several versions, whose reference "
      f"lists are largely the same.")